# 🍌 Open-Vision-Banana — Interactive Colab Demo

Explore **core RGB mapping algorithms** and **inference optimization** without installing FLUX weights.

**Features**:
- Depth → RGB → Depth roundtrip with error heatmap
- Normal → RGB → Normal with angular error visualization
- Instance segmentation decoding (5-stage algorithm demo)
- Inference optimization preset comparison
- Benchmark suite (speed, memory, accuracy)

> ⚠️ **GPU not required** for the core algorithm demos. For full FLUX inference, use `scripts/inference.py` with a GPU.

In [ ]:
# ═══════════════════════════════════════════════════════════
# 1. Setup (run once)
# ═══════════════════════════════════════════════════════════
!pip install -q torch torchvision pillow pyyaml scipy matplotlib

import os
if not os.path.exists('open-vision-banana'):
    !git clone --depth 1 https://github.com/user/open-vision-banana.git
%cd open-vision-banana
!pip install -q -e .

import torch, numpy as np, matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
print('✅ Setup complete')

## 2. Depth Mapping — Power Transform + Spectral Colormap

The paper's key insight: **depth can be encoded as RGB** using a power transform and spectral colormap, then decoded back.

**Math**: `f(d, λ, c) = 1 - (1 - d/(λc))^(λ+1)` with λ=-3, c=10/3

In [ ]:
from open_vision_banana.core import DepthRGBMapper

mapper = DepthRGBMapper()
depth = torch.rand(1, 1, 512, 512) * 50.0 + 0.1

rgb = mapper.depth_to_rgb(depth)
recovered = mapper.rgb_to_depth(rgb)

abs_err = (recovered - depth).abs()
rel_err = abs_err / depth.clamp(min=0.1)

fig = plt.figure(figsize=(16, 4))
gs = GridSpec(1, 4, figure=fig)

ax0 = fig.add_subplot(gs[0, 0])
ax0.imshow(depth.squeeze().numpy(), cmap='viridis', vmin=0, vmax=50)
ax0.set_title('Input Depth (m)'); ax0.axis('off')

ax1 = fig.add_subplot(gs[0, 1])
ax1.imshow(rgb.squeeze().permute(1,2,0).numpy())
ax1.set_title('RGB Encoding (Spectral)'); ax1.axis('off')

ax2 = fig.add_subplot(gs[0, 2])
ax2.imshow(recovered.squeeze().numpy(), cmap='viridis', vmin=0, vmax=50)
ax2.set_title('Recovered Depth'); ax2.axis('off')

ax3 = fig.add_subplot(gs[0, 3])
im = ax3.imshow(rel_err.squeeze().numpy(), cmap='hot', vmin=0, vmax=0.01)
ax3.set_title(f'Relative Error (mean={rel_err.mean():.4f})'); ax3.axis('off')
plt.colorbar(im, ax=ax3, fraction=0.046)

plt.tight_layout(); plt.show()
print(f'📊 Near-depth (<10m) roundtrip MAE: {abs_err[depth<10].mean():.4f} m')
print(f'📊 Overall roundtrip mean rel error:  {rel_err.mean():.4f}')

## 3. Normal Mapping — Linear Encoding with Cosine Similarity

Surface normals encode as RGB via simple linear mapping: `R=(1-x)/2, G=(1+y)/2, B=(1+z)/2`

In [ ]:
from open_vision_banana.core import NormalRGBMapper

mapper_n = NormalRGBMapper()
normals = torch.randn(1, 3, 256, 256)
normals = normals / normals.norm(dim=1, keepdim=True).clamp(min=1e-8)

rgb_n = mapper_n.normal_to_rgb(normals)
rec_n = mapper_n.rgb_to_normal(rgb_n)

cos_sim = (normals * rec_n).sum(dim=1)
ang_err = torch.acos(cos_sim.clamp(-1, 1)) * 180 / 3.14159265

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow((normals.squeeze().permute(1,2,0).numpy() + 1) / 2)
axes[0].set_title('Input Normals'); axes[0].axis('off')
axes[1].imshow(rgb_n.squeeze().permute(1,2,0).numpy())
axes[1].set_title('RGB Encoding'); axes[1].axis('off')
axes[2].imshow((rec_n.squeeze().permute(1,2,0).numpy() + 1) / 2)
axes[2].set_title('Recovered Normals'); axes[2].axis('off')
im = axes[3].imshow(ang_err.squeeze().numpy(), cmap='hot', vmin=0, vmax=2)
axes[3].set_title(f'Angular Error° (mean={ang_err.mean():.2f}°)'); axes[3].axis('off')
plt.colorbar(im, ax=axes[3], fraction=0.046)
plt.tight_layout(); plt.show()
print(f'📊 Cosine similarity: {cos_sim.mean():.6f}')
print(f'📊 Mean angular error: {ang_err.mean():.2f}°')

## 4. Instance Segmentation — 5-Stage Decoding Demo

The paper's Appendix A algorithm, implemented with fully vectorized operations:
1. Background initialization (color threshold)
2. Union-Find color grouping (8-connectivity)
3. Noise pruning (area threshold)
4. Boundary erosion (remove diffusion artifacts)
5. Spatial merging (bbox + color distance)

In [ ]:
from open_vision_banana.core import SegmentationDecoder

decoder = SegmentationDecoder()

# Create synthetic segmentation output (3 colored blobs + background)
H, W = 256, 256
seg_rgb = torch.zeros(1, 3, H, W)

# Blob 1: red
seg_rgb[0, 0, 40:120, 40:120] = 1.0
# Blob 2: green
seg_rgb[0, 1, 140:220, 40:120] = 1.0
# Blob 3: blue
seg_rgb[0, 2, 80:160, 160:240] = 1.0

# Add some noise and boundary artifacts
noise = torch.randn_like(seg_rgb) * 0.05
seg_rgb = (seg_rgb + noise).clamp(0, 1)

instances = decoder.decode_instance(seg_rgb)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(seg_rgb.squeeze().permute(1,2,0).numpy())
axes[0].set_title('Synthetic RGB Output'); axes[0].axis('off')

# Visualize instance masks
mask_overlay = np.zeros((H, W, 3))
colors = [(1,0,0), (0,1,0), (0,0,1)]
for i, inst in enumerate(instances[0]):
    mask = inst['mask'].cpu().numpy().astype(float)
    c = colors[i % len(colors)]
    for ch in range(3):
        mask_overlay[..., ch] += mask * c[ch] * 0.5
mask_overlay = np.clip(mask_overlay, 0, 1)
axes[1].imshow(mask_overlay)
axes[1].set_title(f'Detected Instances: {len(instances[0])}'); axes[1].axis('off')
plt.tight_layout(); plt.show()
print(f'✅ Detected {len(instances[0])} instances (expected: 3)')

## 5. Inference Optimization Presets

Compare memory/speed trade-offs for different hardware configurations.

In [ ]:
from open_vision_banana.inference.optimization import InferenceOptions, estimate_vram_savings

presets = {
    'balanced': InferenceOptions(dtype='float16', enable_model_cpu_offload=True, enable_vae_slicing=True),
    'speed': InferenceOptions(dtype='float16', enable_torch_compile=True, enable_vae_slicing=True),
    'memory': InferenceOptions(dtype='float16', enable_sequential_cpu_offload=True, enable_vae_tiling=True, quantize_mappers_int8=True),
    'quality': InferenceOptions(dtype='bfloat16', enable_torch_compile=True),
    'cpu': InferenceOptions(dtype='float32', enable_sequential_cpu_offload=True),
}

print(f'{"Profile":<12} {"VRAM Impact":<50} {"Use Case":<30}')
print('-' * 100)
for name, opts in presets.items():
    notes = estimate_vram_savings(opts)
    impact = ' | '.join(notes.values())[:48] if notes else 'None'
    use_case = {
        'balanced': 'A10 / RTX 3090 (24GB)',
        'speed': 'A100 / H100 (fastest)',
        'memory': 'RTX 3060 (12GB)',
        'quality': 'A100-80GB (max fidelity)',
        'cpu': 'No GPU (universal)',
    }.get(name, '')
    print(f'{name:<12} {impact:<50} {use_case:<30}')

## 6. Benchmark Suite

Measure speed, memory, and accuracy of core operations.

In [ ]:
import time

def bench(fn, *args, warmup=5, iters=50):
    for _ in range(warmup): fn(*args)
    t0 = time.time()
    for _ in range(iters): fn(*args)
    return (time.time() - t0) / iters * 1000

res = 512
depth = torch.rand(1, 1, res, res) * 50.0
normal = torch.randn(1, 3, res, res)
normal = normal / normal.norm(dim=1, keepdim=True).clamp(min=1e-8)

d_mapper = DepthRGBMapper()
n_mapper = NormalRGBMapper()
decoder = SegmentationDecoder()

results = {
    'Depth encode': bench(d_mapper.depth_to_rgb, depth),
    'Depth decode': bench(d_mapper.rgb_to_depth, d_mapper.depth_to_rgb(depth)),
    'Normal encode': bench(n_mapper.normal_to_rgb, normal),
    'Normal decode': bench(n_mapper.rgb_to_normal, n_mapper.normal_to_rgb(normal)),
    'Instance decode': bench(decoder.decode_instance, torch.rand(1,3,128,128)),
}

print(f'{"Operation":<25} {"Median ms":>12}')
print('-' * 40)
for op, ms in results.items():
    print(f'{op:<25} {ms:>12.2f}')

print(f'\n📊 Total roundtrip (depth+normal): {results["Depth encode"]+results["Depth decode"]+results["Normal encode"]+results["Normal decode"]:.2f} ms')